In [ ]:
import os
import nibabel as nib
import numpy as np

input_dirs = [
    '/home/amenacer/Stage/Data/Segmentation/resultas/Tollsome_output1',
    '/home/amenacer/Stage/Data/Segmentation/resultas/Tollsome_output2',
    '/home/amenacer/Stage/Data/Segmentation/resultas/Tollsome_output3'
]
output_dir = '/home/amenacer/Stage/Data/Segmentation/resultas/split_volumes'
os.makedirs(output_dir, exist_ok=True)

for input_dir in input_dirs:
    group_name = os.path.basename(input_dir)
    group_output_dir = os.path.join(output_dir, group_name)
    os.makedirs(group_output_dir, exist_ok=True)

    for filename in os.listdir(input_dir):
        if filename.endswith('.nii.gz'):
            filepath = os.path.join(input_dir, filename)
            img = nib.load(filepath)
            data = img.get_fdata()
            affine = img.affine

            num_frames = data.shape[-1]
            frames_per_split = max(num_frames // 30, 1)

            basename = filename.replace('.nii.gz', '')
            sub_output_dir = os.path.join(group_output_dir, basename)
            os.makedirs(sub_output_dir, exist_ok=True)

            for i in range(30):
                start_frame = i * frames_per_split
                end_frame = start_frame + frames_per_split

                if i == 29 or end_frame > num_frames:
                    end_frame = num_frames

                split_data = data[..., start_frame:end_frame]
                split_img = nib.Nifti1Image(split_data, affine)

                split_filename = f'{basename}_part{i+1}.nii.gz'
                split_filepath = os.path.join(sub_output_dir, split_filename)
                nib.save(split_img, split_filepath)

print("Découpage terminé !")

In [ ]:
import os
dossiers = [
    '/home/amenacer/Stage/Data/Segmentation/resultas/Tollsome_output1',
    '/home/amenacer/Stage/Data/Segmentation/resultas/Tollsome_output2',
    '/home/amenacer/Stage/Data/Segmentation/resultas/Tollsome_output3'
]

for dossier in dossiers:
    if os.path.exists(dossier):
        print(f"{dossier} existe.")
        print("Contenu:", os.listdir(dossier))
    else:
        print(f"{dossier} n'existe pas.")


In [ ]:
import os
import nibabel as nib
import numpy as np

# Chemins des dossiers contenant les segmentations découpées
segmentation_dirs = [
    '/home/amenacer/Stage/Data/Segmentation/resultas/split_volumes/Tollsome_output1',
    '/home/amenacer/Stage/Data/Segmentation/resultas/split_volumes/Tollsome_output2',
    '/home/amenacer/Stage/Data/Segmentation/resultas/split_volumes/Tollsome_output3'
]

# Dossier de sauvegarde des résultats
areas_output_dir = '/home/amenacer/Stage/Data/Segmentation/resultas/segmented_areas'
os.makedirs(areas_output_dir, exist_ok=True)

# Extraction des aires segmentées
for seg_dir in segmentation_dirs:
    group_name = os.path.basename(seg_dir)
    group_output_dir = os.path.join(areas_output_dir, group_name)
    os.makedirs(group_output_dir, exist_ok=True)
    print(f'Traitement du dossier : {seg_dir}')

    for sub_folder in os.listdir(seg_dir):
        sub_folder_path = os.path.join(seg_dir, sub_folder)
        if os.path.isdir(sub_folder_path):
            print(f'Traitement du sous-dossier : {sub_folder_path}')

            for filename in os.listdir(sub_folder_path):
                if filename.endswith('.nii.gz'):
                    filepath = os.path.join(sub_folder_path, filename)
                    print(f'Chargement du fichier : {filepath}')
                    img = nib.load(filepath)
                    data = img.get_fdata()

                    # Calcul des aires pour chaque instant temporel
                    areas = np.sum(data > 0, axis=(0, 1))  # Compte les pixels segmentés à chaque temps

                    # Sauvegarde des aires dans un fichier texte
                    basename = filename.replace('.nii.gz', '')
                    area_filename = f'{basename}_areas.txt'
                    area_subdir = os.path.join(group_output_dir, sub_folder)
                    os.makedirs(area_subdir, exist_ok=True)
                    area_filepath = os.path.join(area_subdir, area_filename)
                    np.savetxt(area_filepath, areas, fmt='%d')
                    print(f'Sauvegardé : {area_filepath}')

print("Extraction des aires terminée !")


J'ai préparé le script pour construire les descripteurs à partir des aires segmentées. Il calcule précisément les descripteurs suivants pour chaque série temporelle :

    Aire minimale

    Aire maximale

    Aire moyenne

    Déviation standard des aires

    Pente ascendante maximale

    Pente descendante maximale

In [ ]:
import os
import numpy as np

# Dossier contenant les aires segmentées
areas_dir = '/home/amenacer/Stage/Data/Segmentation/resultas/segmented_areas'

# Dossier pour sauvegarder les descripteurs
features_output_dir = '/home/amenacer/Stage/Data/Segmentation/resultas/features'
os.makedirs(features_output_dir, exist_ok=True)

# Calcul des descripteurs
for group_name in os.listdir(areas_dir):
    group_path = os.path.join(areas_dir, group_name)
    group_features_dir = os.path.join(features_output_dir, group_name)
    os.makedirs(group_features_dir, exist_ok=True)

    for sub_folder in os.listdir(group_path):
        sub_folder_path = os.path.join(group_path, sub_folder)

        for area_file in os.listdir(sub_folder_path):
            if area_file.endswith('_areas.txt'):
                areas_path = os.path.join(sub_folder_path, area_file)
                areas = np.loadtxt(areas_path)

                if areas.size == 0:
                    print(f"Le fichier {areas_path} est vide. Ignoré.")
                    continue  # passer au fichier suivant

                # Calcul des descripteurs
                min_area = np.min(areas)
                max_area = np.max(areas)
                mean_area = np.mean(areas)
                std_area = np.std(areas)

                if areas.size > 1:
                    ascending_slope = np.max(np.diff(areas))
                    descending_slope = np.min(np.diff(areas))
                else:
                    ascending_slope = 0
                    descending_slope = 0

                features = np.array([
                    min_area, 
                    max_area, 
                    mean_area, 
                    std_area, 
                    ascending_slope, 
                    descending_slope
                ])

                # Sauvegarde des descripteurs
                features_filename = area_file.replace('_areas.txt', '_features.txt')
                features_filepath = os.path.join(group_features_dir, features_filename)
                np.savetxt(features_filepath, features, 
                           header='min_area max_area mean_area std_area ascending_slope descending_slope', 
                           fmt='%.4f')
                
                print(f'Descripteurs sauvegardés : {features_filepath}')

print("Calcul des descripteurs terminé !")


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Chemin vers tes descripteurs
features_output_dir = '/home/amenacer/Stage/Data/Segmentation/resultas/features'

# Création d'une DataFrame vide pour regrouper toutes les données
all_features = []

# Lecture des fichiers
for group_name in os.listdir(features_output_dir):
    group_path = os.path.join(features_output_dir, group_name)

    for feature_file in os.listdir(group_path):
        if feature_file.endswith('_features.txt'):
            file_path = os.path.join(group_path, feature_file)
            features = np.loadtxt(file_path)

            all_features.append({
                'Groupe': group_name,
                'Fichier': feature_file.replace('_features.txt', ''),
                'Min_Area': features[0],
                'Max_Area': features[1],
                'Mean_Area': features[2],
                'Std_Area': features[3],
                'Ascending_Slope': features[4],
                'Descending_Slope': features[5]
            })

# Conversion en DataFrame
features_df = pd.DataFrame(all_features)

# Sauvegarde en csv pour usage ultérieur si nécessaire
features_df.to_csv('/home/amenacer/Stage/Data/Segmentation/resultas/features_summary.csv', index=False)

# Visualisation claire des descripteurs avec seaborn
plt.figure(figsize=(14, 8))
sns.boxplot(data=features_df, x='Groupe', y='Mean_Area')
plt.title('Distribution des aires moyennes par groupe')
plt.ylabel('Aire moyenne')
plt.xlabel('Groupe')
plt.grid(True)
plt.show()

# Visualisation complète de tous les descripteurs avec pairplot
sns.pairplot(features_df, hue='Groupe', diag_kind='kde')
plt.suptitle('Visualisation complète des descripteurs', y=1.02)
plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Chemin vers tes descripteurs
features_output_dir = '/home/amenacer/Stage/Data/Segmentation/resultas/features'

# Préparation des données
all_features = []

for group_name in os.listdir(features_output_dir):
    group_path = os.path.join(features_output_dir, group_name)
    
    # Vérifie si c'est bien un dossier
    if not os.path.isdir(group_path):
        continue
    
    for feature_file in os.listdir(group_path):
        if feature_file.endswith('_features.txt'):
            file_path = os.path.join(group_path, feature_file)
            features = np.loadtxt(file_path)

            all_features.append({
                'Groupe': group_name,
                'Fichier': feature_file.replace('_features.txt', ''),
                'Min_Area': features[0],
                'Max_Area': features[1],
                'Mean_Area': features[2],
                'Std_Area': features[3],
                'Ascending_Slope': features[4],
                'Descending_Slope': features[5]
            })

# Conversion en DataFrame
features_df = pd.DataFrame(all_features)

# Sauvegarde CSV (optionnelle)
features_df.to_csv('/home/amenacer/Stage/Data/Segmentation/resultas/features_summary.csv', index=False)

# Liste des features à visualiser
features_list = ['Min_Area', 'Max_Area', 'Mean_Area', 'Std_Area', 'Ascending_Slope', 'Descending_Slope']

# Visualisation individuelle claire pour chaque feature
for feature in features_list:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='Groupe', y=feature, data=features_df)
    plt.title(f'Distribution du descripteur : {feature}', fontsize=15)
    plt.xlabel('Groupe', fontsize=12)
    plt.ylabel(feature, fontsize=12)
    plt.grid(True)
    plt.show()
